# BrainIAC features for the OASIS-1 linear probe

This notebook runs `scripts/brainiac_features.py` of the `oasis1-cognitive-baseline` repository on Google Colab:
it registers the FreeSurfer `brainmask.mgz` of the 226 study subjects to the BrainIAC template and extracts
frozen BrainIAC features (ViT-B backbone of the public checkpoint). Nothing is trained here. The linear probe
itself is fitted locally with `scripts/probe.py`.

**Before running**
1. The OASIS-1 FreeSurfer release must be in Google Drive under `MyDrive/OASIS_Freesurfer/` (as for the other notebooks).
2. Download the BrainIAC checkpoint bundle from the link in the [BrainIAC README](https://github.com/AIM-KannLab/BrainIAC)
   (Dropbox, about 7.4 GB zipped), extract `BrainIAC.ckpt` and place it at `MyDrive/BrainIAC/BrainIAC.ckpt`.
3. BrainIAC is licensed for non-commercial academic research use only.

Stage 1 (preprocessing) needs no GPU and takes roughly 5-20 s per subject on Colab CPUs; it resumes with `--skip-existing`.
Stage 2 (features) is best run on a GPU runtime and takes a few minutes.

The feature CSVs are derived from OASIS data and stay in Drive; they are not part of the repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
OASIS_DIR = f'{DRIVE_ROOT}/OASIS_Freesurfer'               # discs 1-11 of the FreeSurfer release
CHECKPOINT = f'{DRIVE_ROOT}/BrainIAC/BrainIAC.ckpt'        # downloaded from the BrainIAC Dropbox link
OUT_DIR = f'{DRIVE_ROOT}/alzheimer_anfis_outputs/brainiac_probe'  # registered volumes, QC montage, feature CSVs

import os
assert os.path.isdir(OASIS_DIR), OASIS_DIR
assert os.path.isfile(CHECKPOINT), f'{CHECKPOINT} not found: download BrainIAC.ckpt first'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
%cd /content
!rm -rf oasis1-cognitive-baseline BrainIAC
!git clone -q https://github.com/Yuliawang520/oasis1-cognitive-baseline.git
!git clone -q https://github.com/AIM-KannLab/BrainIAC.git
!pip install -q monai SimpleITK nibabel
import monai, SimpleITK, torch
print('monai', monai.__version__, '| SimpleITK', SimpleITK.Version_VersionString(), '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Stage 1: N4 and rigid registration to the BrainIAC template (CPU)

HD-BET is skipped because `brainmask.mgz` is already skull-stripped. Re-run this cell after a disconnect; it resumes.

In [ ]:
!python /content/oasis1-cognitive-baseline/scripts/brainiac_features.py \
    --oasis-dir "$OASIS_DIR" \
    --folds /content/oasis1-cognitive-baseline/data/folds.csv \
    --brainiac-src /content/BrainIAC/src \
    --checkpoint "$CHECKPOINT" \
    --out-dir "$OUT_DIR" \
    --stage preprocess --skip-existing

In [ ]:
# Visual check of the registration: mid-axial (top) and mid-coronal (bottom) slices of the first twelve subjects.
from IPython.display import Image, display
display(Image(filename=f'{OUT_DIR}/qc_registration.png'))
import pandas as pd
m = pd.read_csv(f'{OUT_DIR}/registration_metric.csv')
print(m.describe())
print('least well registered (highest Mattes MI):')
print(m.sort_values('mattes_mi', ascending=False).head(5).to_string(index=False))

## Stage 2: frozen BrainIAC features (GPU runtime recommended)

Writes `brainiac_features.csv` (BrainIAC's own representation, the first token of the final layer) and
`brainiac_features_meanpool.csv` (mean of the 216 patch tokens, a sensitivity variant) to `OUT_DIR`.

In [ ]:
!python /content/oasis1-cognitive-baseline/scripts/brainiac_features.py \
    --oasis-dir "$OASIS_DIR" \
    --folds /content/oasis1-cognitive-baseline/data/folds.csv \
    --brainiac-src /content/BrainIAC/src \
    --checkpoint "$CHECKPOINT" \
    --out-dir "$OUT_DIR" \
    --stage features
!ls -la "$OUT_DIR"

## Next step (local)

Download the two feature CSVs from `OUT_DIR` and run, in the repository:

```bash
python scripts/probe.py --cohort path/to/oasis_master_features_labels.csv --features brainiac_features.csv --name "BrainIAC probe"
python scripts/probe.py --cohort path/to/oasis_master_features_labels.csv --features brainiac_features_meanpool.csv --name "BrainIAC probe (mean pool)"
```

The out-of-fold probe scores (`data/oof_probe_*.csv`) can be released; the feature CSVs cannot.